# Practical 6 – Convolutional Neural Network from Scratch
**Project:** NoteReader – Indian Currency Speaker for the Blind  
**Institute:** RCOEM, Nagpur – CSE Deep Learning Lab  
**Mapped to:** P6 (CNN Architectures)  

## Objectives
1. Understand convolution, pooling and flatten mathematically with a NumPy toy demo.
2. Build a CNN baseline (no pre-training) for Indian currency recognition.
3. Understand common viva questions: *What is a convolution? What is pooling? Why Flatten?*

## Viva Q&A (Markdown – read before viva)

### Q1 – What is a convolution operation?
A convolution slides a small learnable filter (kernel) over the input image pixel by pixel.  
At each position it computes the **dot product** of the filter weights and the local patch of pixels,  
producing a single number in the output feature map.  
Mathematically for a 2-D input I and kernel K:  
```
output[i,j] = Σ_m Σ_n  I[i+m, j+n] * K[m, n]
```
The filter learns to detect edges, colours, textures – and in deeper layers, entire shapes like a corner of a note.

### Q2 – What is pooling?
Pooling (usually Max-Pooling 2×2, stride 2) reduces the spatial size of the feature map by keeping  
only the *maximum* value inside each 2×2 window. Benefits:
- Reduces computation and memory.
- Introduces slight translation invariance (the note doesn't have to be perfectly centred).
- Acts as a mild regulariser by discarding less-active neurons.

### Q3 – Why do we need Flatten?
After the convolutional and pooling layers the data is still a 3-D tensor  
(height × width × channels). A Dense (fully-connected) layer expects a 1-D vector.  
Flatten simply reshapes (H × W × C) → (H*W*C,) so that every feature can connect to every neuron in the next layer.

### Q4 – What is softmax?
Softmax converts the raw logit scores of the last Dense layer into a probability distribution  
that sums to 1. For class k out of K total classes:  
```
softmax(z_k) = exp(z_k) / Σ_{j=1}^{K} exp(z_j)
```
The class with the highest probability is the predicted denomination.

## Section 1 – NumPy Forward-Pass Demo (as in P6 manual)
This is the *exact* type of step-by-step calculation asked in the practical manual and in the viva.  
We use:
- 4×4 input
- 3×3 kernel, stride 1, no padding → 2×2 output
- Flatten → 4 values
- Single sigmoid neuron → scalar output
- MSE loss

In [ ]:
"""
P6 NumPy Demo – Forward Propagation Step by Step
===================================================
This mirrors the Practical 6 manual exercise so you can explain
every number in the viva.
"""

import numpy as np

# ── 1. Define a tiny 4×4 input (grayscale, one channel) ──────────────────────
X = np.array([
    [1, 2, 3, 0],
    [0, 1, 2, 3],
    [3, 0, 1, 2],
    [2, 3, 0, 1]
], dtype=float)
print("Input (4×4):\n", X)

# ── 2. Define a 3×3 kernel (edge detector-style) ─────────────────────────────
K = np.array([
    [ 1,  0, -1],
    [ 1,  0, -1],
    [ 1,  0, -1]
], dtype=float)
print("\nKernel (3×3):\n", K)

# ── 3. Convolution: stride=1, padding=0 → output is (4-3+1) × (4-3+1) = 2×2 ─
def conv2d(image, kernel, stride=1):
    """Manual 2-D convolution (cross-correlation as used in CNNs)."""
    out_h = (image.shape[0] - kernel.shape[0]) // stride + 1
    out_w = (image.shape[1] - kernel.shape[1]) // stride + 1
    output = np.zeros((out_h, out_w))
    for i in range(out_h):
        for j in range(out_w):
            patch = image[i*stride:i*stride+kernel.shape[0],
                          j*stride:j*stride+kernel.shape[1]]
            # Dot product of patch and kernel
            output[i, j] = np.sum(patch * kernel)
    return output

feature_map = conv2d(X, K)
print("\nFeature Map after Conv2D (2×2):\n", feature_map)

# ── 4. ReLU activation (zero out negatives) ──────────────────────────────────
relu_map = np.maximum(0, feature_map)
print("\nAfter ReLU:\n", relu_map)

# ── 5. Max Pooling 2×2 (the whole 2×2 map is one window here) ────────────────
# In practice pooling runs over a larger feature map. Here we just take the max.
pooled = np.array([[np.max(relu_map)]])
print("\nAfter MaxPool (1×1):", pooled)

# ── 6. Flatten → 1-D vector ──────────────────────────────────────────────────
flat = feature_map.flatten()   # use pre-ReLU map for richer numbers in demo
print("\nFlattened (4 values):", flat)

# ── 7. Fully-connected layer: 4 → 1 sigmoid neuron ──────────────────────────
# Weights and bias (fixed for demo, would be learned during training)
W = np.array([0.5, -0.3, 0.8, 0.1])
b = 0.2
z = np.dot(W, flat) + b          # linear combination
print("\nLinear combination z = W·x + b =", z)

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

y_hat = sigmoid(z)
print("Sigmoid output ŷ =", round(y_hat, 4))

# ── 8. MSE Loss ──────────────────────────────────────────────────────────────
y_true = 1.0   # suppose the note IS present (binary for demo)
mse = (y_hat - y_true) ** 2
print("MSE Loss =", round(mse, 4))
print("\n✓ This is exactly the P6 manual exercise replicated in NumPy.")

## Section 2 – Dataset Setup
We use the **Indian Currency Note Dataset** from Kaggle.

### Kaggle Setup Instructions
```bash
pip install kaggle
# Place kaggle.json (from https://www.kaggle.com/settings → API) in ~/.kaggle/
kaggle datasets download -d mishra1993/indian-currency-dataset
# OR search: kaggle datasets list -s "indian currency notes"
```

### If a class is missing or has fewer than 50 images
The table below lists how many images are needed and how to capture them:

| Class | Minimum images | Notes |
|-------|---------------|-------|
| ₹10   | 80 | Very common; capture old brown notes and new chocolate-brown ones |
| ₹20   | 80 | Capture both old (orange) and new (yellow-green) design |
| ₹50   | 80 | – |
| ₹100  | 80 | – |
| ₹200  | 80 | New note; capture under-lit conditions |
| ₹500  | 80 | Capture both sides; often very worn |

### Photo Capture Guide (for blind-user realism)
A blind user *cannot* frame the note perfectly. Capture images that reflect this:

1. **Lighting**: bright sunlight, indoor tube light, single lamp (one side lit), dim room, phone torch.
2. **Note condition**: crisp, folded in half, crumpled, slightly torn edge, old worn notes, new notes.
3. **Partial view**: only 60 % of the note visible (the user may not centre it).
4. **Both sides**: front (portrait side) and back (numeral/motif side).
5. **Held in hand**: fingers partially covering the note.
6. **Background**: plain table, cluttered bag, patterned cloth, wallet interior.
7. **Blur / shake**: slight motion blur, slightly out-of-focus.
8. **Angle**: 0°, ±15°, ±30° rotation; slight perspective warp (note held at arm's length).

Capture **at least 80 images per class** (ideally 150+). Name them `<denomination>_<seq>.jpg`  
and place in `dataset/raw/<denomination>/`.

In [ ]:
"""
Dataset loading, augmentation and split.
Run this cell AFTER placing images in dataset/raw/<denomination>/
"""

import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array

# ── Config ────────────────────────────────────────────────────────────────────
DATA_DIR   = Path('../dataset/raw')          # root: sub-folders named by denomination
PROC_DIR   = Path('../dataset/processed')   # will hold train/val/test splits
IMG_SIZE   = (224, 224)                      # MobileNetV2 canonical size
SEED       = 42                              # fix for reproducibility across all practicals
CLASSES    = ['10', '20', '50', '100', '200', '500']

np.random.seed(SEED)
tf.random.set_seed(SEED)

# ── 1. Count images per class ─────────────────────────────────────────────────
print("=" * 50)
print("CLASS DISTRIBUTION (raw)")
print("=" * 50)
counts = {}
for cls in CLASSES:
    folder = DATA_DIR / cls
    n = len(list(folder.glob('*.jpg')) + list(folder.glob('*.png'))) if folder.exists() else 0
    counts[cls] = n
    status = '✓' if n >= 80 else f'⚠ NEED {80-n} MORE'
    print(f"  ₹{cls:>4} → {n:>5} images  {status}")

# Plot
plt.figure(figsize=(8, 4))
bars = plt.bar([f'₹{c}' for c in CLASSES], counts.values(), color='steelblue', edgecolor='white')
plt.axhline(80, color='red', linestyle='--', label='Minimum (80)')
plt.title('Dataset – Images per Class')
plt.xlabel('Denomination'); plt.ylabel('Image count')
plt.legend(); plt.tight_layout()
os.makedirs('../results/plots', exist_ok=True)
plt.savefig('../results/plots/class_distribution.png', dpi=150)
plt.show()

# ── 2. Load + resize all images into numpy arrays ─────────────────────────────
X_all, y_all = [], []
for idx, cls in enumerate(CLASSES):
    folder = DATA_DIR / cls
    if not folder.exists():
        print(f'[SKIP] {cls} folder not found')
        continue
    for img_path in list(folder.glob('*.jpg')) + list(folder.glob('*.png')):
        try:
            img = load_img(img_path, target_size=IMG_SIZE)
            X_all.append(img_to_array(img))
            y_all.append(idx)
        except Exception as e:
            print(f'[WARN] Could not load {img_path}: {e}')

X_all = np.array(X_all, dtype='float32')
y_all = np.array(y_all)
print(f"\nTotal images loaded: {len(X_all)}, shape: {X_all.shape}")

# ── 3. 70 / 15 / 15 split (SAME SPLIT USED IN ALL PRACTICALS) ─────────────────
X_temp, X_test, y_temp, y_test = train_test_split(
    X_all, y_all, test_size=0.15, random_state=SEED, stratify=y_all)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.15/(0.70+0.15), random_state=SEED, stratify=y_temp)

print(f"Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}")
np.save('../dataset/X_train.npy', X_train); np.save('../dataset/y_train.npy', y_train)
np.save('../dataset/X_val.npy',   X_val);   np.save('../dataset/y_val.npy',   y_val)
np.save('../dataset/X_test.npy',  X_test);  np.save('../dataset/y_test.npy',  y_test)
print("Saved numpy splits to dataset/")

# ── 4. Show sample images ─────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 6, figsize=(16, 6))
for col, cls in enumerate(CLASSES):
    cls_idx = list(y_train).index(col) if col in y_train else None
    if cls_idx is not None:
        for row in range(2):
            idxs = np.where(y_train == col)[0]
            if len(idxs) > row:
                axes[row, col].imshow(X_train[idxs[row]].astype('uint8'))
                axes[row, col].set_title(f'₹{cls}')
            axes[row, col].axis('off')
plt.suptitle('Sample Training Images (2 per class)')
plt.tight_layout()
plt.savefig('../results/plots/sample_images.png', dpi=150)
plt.show()

## Section 3 – CNN Baseline Architecture

In [ ]:
"""
CNN from Scratch – Baseline Model
===================================
Architecture: Conv→Pool→Conv→Pool→Conv→Pool→Flatten→Dense→Dropout→Softmax
WHY each layer is here is answered in the viva Q&A markdown above.
"""

from tensorflow import keras
from tensorflow.keras import layers, Sequential
from tensorflow.keras.callbacks import ModelCheckpoint, ReduceLROnPlateau

NUM_CLASSES = 6
BATCH_SIZE  = 32
EPOCHS_CNN  = 30

# ── Normalise pixels to [0,1] (simple normalisation for the scratch CNN) ───────
X_train_n = X_train / 255.0
X_val_n   = X_val   / 255.0
X_test_n  = X_test  / 255.0

# ── One-hot encode labels ──────────────────────────────────────────────────────
y_train_oh = keras.utils.to_categorical(y_train, NUM_CLASSES)
y_val_oh   = keras.utils.to_categorical(y_val,   NUM_CLASSES)
y_test_oh  = keras.utils.to_categorical(y_test,  NUM_CLASSES)

# ── Data augmentation (applied only to training set) ──────────────────────────
# WHY: our dataset is small; augmentation synthetically enlarges it by showing
#      the network rotated, zoomed and shifted versions of the same note.
datagen = ImageDataGenerator(
    rotation_range=30,         # random rotation ±30°
    zoom_range=0.2,            # random zoom
    brightness_range=[0.6, 1.4], # simulate different lighting
    width_shift_range=0.15,    # horizontal shift
    height_shift_range=0.15,   # vertical shift
    horizontal_flip=True,      # notes can appear either way
    shear_range=10,            # slight perspective warp
    fill_mode='nearest'
)
datagen.fit(X_train_n)

def build_cnn_scratch(input_shape=(224, 224, 3), num_classes=6):
    """
    Build a CNN from scratch.
    The increasing filter count (32→64→128) follows the standard practice:
    early layers detect edges/colours, later layers detect complex patterns.
    """
    model = Sequential([
        # Block 1
        layers.Conv2D(32, (3,3), activation='relu', padding='same',
                      input_shape=input_shape, name='conv1'),
        layers.MaxPooling2D((2,2), name='pool1'),

        # Block 2
        layers.Conv2D(64, (3,3), activation='relu', padding='same', name='conv2'),
        layers.MaxPooling2D((2,2), name='pool2'),

        # Block 3
        layers.Conv2D(128, (3,3), activation='relu', padding='same', name='conv3'),
        layers.MaxPooling2D((2,2), name='pool3'),

        # Flatten: convert 3-D feature maps to 1-D vector for Dense layer
        layers.Flatten(name='flatten'),

        # Fully-connected head
        layers.Dense(256, activation='relu', name='fc1'),
        layers.Dropout(0.4, name='dropout'),          # mild regularisation
        layers.Dense(num_classes, activation='softmax', name='output')
    ], name='CNN_Scratch')
    return model

cnn_model = build_cnn_scratch()
cnn_model.summary()

# ── Compile with Adam (default; optimizers will be compared in P4 notebook) ───
cnn_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# ── Callbacks ──────────────────────────────────────────────────────────────────
ckpt = ModelCheckpoint('../models/cnn_scratch_best.h5',
                        monitor='val_accuracy', save_best_only=True, verbose=1)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, verbose=1)

# ── Train ─────────────────────────────────────────────────────────────────────
history_cnn = cnn_model.fit(
    datagen.flow(X_train_n, y_train_oh, batch_size=BATCH_SIZE, seed=SEED),
    epochs=EPOCHS_CNN,
    validation_data=(X_val_n, y_val_oh),
    callbacks=[ckpt, reduce_lr],
    verbose=1
)

# ── Load best weights & evaluate ──────────────────────────────────────────────
cnn_model.load_weights('../models/cnn_scratch_best.h5')
loss_cnn, acc_cnn = cnn_model.evaluate(X_test_n, y_test_oh, verbose=0)
print(f"\nCNN Scratch – Test Accuracy: {acc_cnn*100:.2f}%  |  Test Loss: {loss_cnn:.4f}")
cnn_model.save('../models/cnn_scratch.h5')

In [ ]:
"""
Plot training curves for CNN Scratch.
Saved to results/plots/ for the report and for the Model Comparison screen in the app.
"""
import json

def plot_history(history, title, save_name):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
    ax1.plot(history.history['loss'],     label='Train Loss', color='royalblue')
    ax1.plot(history.history['val_loss'], label='Val Loss',   color='salmon')
    ax1.set(title=f'{title} – Loss', xlabel='Epoch', ylabel='Loss')
    ax1.legend()

    ax2.plot(history.history['accuracy'],     label='Train Acc', color='royalblue')
    ax2.plot(history.history['val_accuracy'], label='Val Acc',   color='salmon')
    ax2.set(title=f'{title} – Accuracy', xlabel='Epoch', ylabel='Accuracy')
    ax2.legend()

    plt.tight_layout()
    plt.savefig(f'../results/plots/{save_name}.png', dpi=150)
    plt.show()

plot_history(history_cnn, 'CNN Scratch', 'cnn_scratch_training')

# ── Per-class report ──────────────────────────────────────────────────────────
from sklearn.metrics import classification_report, confusion_matrix

y_pred_probs = cnn_model.predict(X_test_n)
y_pred = np.argmax(y_pred_probs, axis=1)
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=[f'₹{c}' for c in CLASSES]))

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=[f'₹{c}' for c in CLASSES],
            yticklabels=[f'₹{c}' for c in CLASSES])
plt.title('CNN Scratch – Confusion Matrix')
plt.ylabel('True'); plt.xlabel('Predicted')
plt.tight_layout()
plt.savefig('../results/plots/cnn_scratch_cm.png', dpi=150)
plt.show()

# Save history for comparison later
with open('../results/cnn_scratch_history.json', 'w') as f:
    json.dump({k: [float(v) for v in vals]
               for k, vals in history_cnn.history.items()}, f)